# 💡 Bước 5: Prescriptive Analytics
Đề xuất chiến lược kinh doanh từ dữ liệu

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
CHARTS = '../output/charts'

try:
    df = pd.read_pickle('../output/df_clean.pkl')
except:
    df = pd.read_csv('../data/shopping_trends.csv')
    for col in ['Gender','Category','Season','Subscription Status','Location','Item Purchased','Frequency of Purchases']:
        df[col] = df[col].astype('category')
    bins = [0,18,25,35,45,55,100]
    df['Age Group'] = pd.cut(df['Age'], bins=bins, labels=['<18','18-25','26-35','36-45','46-55','56+'])

print("✅ Dữ liệu:", df.shape)


## 5.1 Phân khúc khách hàng (Customer Segmentation)

In [ ]:
freq_map = {'Weekly':52,'Bi-Weekly':26,'Fortnightly':26,
            'Monthly':12,'Quarterly':4,'Annually':1,'Every 3 Months':4}
df['Freq_num'] = df['Frequency of Purchases'].astype(str).map(freq_map).fillna(4)

segment = df.groupby('Customer ID').agg(
    Total_Spend=('Purchase Amount (USD)', 'sum'),
    Frequency=('Freq_num', 'first'),
    Subscribed=('Subscription Status', lambda x: 1 if x.iloc[0]=='Yes' else 0),
).reset_index()

q75 = segment['Total_Spend'].quantile(0.75)
q50 = segment['Total_Spend'].quantile(0.50)
segment['Segment'] = segment['Total_Spend'].apply(
    lambda x: 'VIP' if x >= q75 else ('Tiềm năng' if x >= q50 else 'Thông thường')
)
display(segment['Segment'].value_counts().rename('Số lượng').to_frame())

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
segment['Segment'].value_counts().plot(kind='pie', autopct='%1.1f%%',
    ax=axes[0], colors=sns.color_palette('Set2', 3), startangle=90)
axes[0].set_title('Tỷ Trọng Phân Khúc Khách Hàng'); axes[0].set_ylabel('')

seg_rev = segment.groupby('Segment')['Total_Spend'].mean().reset_index()
axes[1].bar(seg_rev['Segment'], seg_rev['Total_Spend'],
            color=sns.color_palette('Set2', 3))
axes[1].set_title('Chi Tiêu TB theo Phân Khúc'); axes[1].set_ylabel('USD')

plt.tight_layout()
plt.savefig(f'{CHARTS}/nb_12_segments.png', dpi=150, bbox_inches='tight')
plt.show()


## 5.2 Sản phẩm nên quảng bá

In [ ]:
prod = df.groupby('Item Purchased', observed=True).agg(
    Revenue=('Purchase Amount (USD)', 'sum'),
    Rating=('Review Rating', 'mean'),
    Count=('Customer ID', 'count')
).reset_index().round(2)

prod['Score'] = (
    (prod['Revenue'] - prod['Revenue'].min()) / (prod['Revenue'].max() - prod['Revenue'].min()) * 0.6 +
    (prod['Rating']  - prod['Rating'].min())  / (prod['Rating'].max()  - prod['Rating'].min())  * 0.4
).round(3)
top_prod = prod.sort_values('Score', ascending=False).head(10)
display(top_prod[['Item Purchased','Revenue','Rating','Score']])

fig, ax = plt.subplots(figsize=(11, 6))
sc = ax.scatter(top_prod['Revenue'], top_prod['Rating'],
                s=top_prod['Count']*3, alpha=0.7,
                c=top_prod['Score'], cmap='YlOrRd')
for _, row in top_prod.iterrows():
    ax.annotate(str(row['Item Purchased']), (row['Revenue'], row['Rating']),
                fontsize=8, ha='left', va='bottom')
ax.set_xlabel('Tổng Doanh Thu (USD)'); ax.set_ylabel('Điểm Đánh Giá TB')
ax.set_title('Bubble Chart: Sản phẩm theo Doanh Thu & Đánh Giá', fontsize=12, fontweight='bold')
plt.colorbar(sc, label='Score tổng hợp')
plt.tight_layout()
plt.savefig(f'{CHARTS}/nb_13_product_score.png', dpi=150, bbox_inches='tight')
plt.show()


## 5.3 Tồn kho theo mùa

In [ ]:
season_cat = df.groupby(['Season','Category'], observed=True).size().reset_index(name='Count')
season_cat['Season']   = season_cat['Season'].astype(str)
season_cat['Category'] = season_cat['Category'].astype(str)
pivot = season_cat.pivot(index='Category', columns='Season', values='Count').fillna(0)

fig, ax = plt.subplots(figsize=(11, 5))
pivot.plot(kind='bar', ax=ax, color=sns.color_palette('Set2', 4))
ax.set_title('Nhu Cầu theo Danh Mục & Mùa (Tối Ưu Tồn Kho)', fontsize=12, fontweight='bold')
ax.set_ylabel('Số Lượt Mua'); ax.set_xlabel('Danh Mục')
plt.xticks(rotation=15); ax.legend(title='Mùa')
plt.tight_layout()
plt.savefig(f'{CHARTS}/nb_14_inventory.png', dpi=150, bbox_inches='tight')
plt.show()


## 5.4 Đề xuất phân bổ ngân sách Marketing

In [ ]:
cat_rev = df.groupby('Category', observed=True)['Purchase Amount (USD)'].sum()
budget  = (cat_rev / cat_rev.sum() * 100).round(2).reset_index()
budget.columns = ['Category', 'Budget_%']
budget['Category'] = budget['Category'].astype(str)
display(budget)

fig, ax = plt.subplots(figsize=(9, 5))
ax.pie(budget['Budget_%'], labels=budget['Category'],
       autopct='%1.1f%%', startangle=90,
       colors=sns.color_palette('Set3', len(budget)))
ax.set_title('Đề Xuất Phân Bổ Ngân Sách Marketing', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{CHARTS}/nb_15_budget.png', dpi=150, bbox_inches='tight')
plt.show()


## 5.5 Tổng hợp Insights & Recommendations

In [ ]:
total_rev    = df['Purchase Amount (USD)'].sum()
top_category = str(df.groupby('Category', observed=True)['Purchase Amount (USD)'].sum().idxmax())
top_season   = str(df.groupby('Season', observed=True)['Purchase Amount (USD)'].sum().idxmax())
top_item     = str(df.groupby('Item Purchased', observed=True)['Purchase Amount (USD)'].sum().idxmax())
top_location = str(df.groupby('Location', observed=True)['Purchase Amount (USD)'].sum().idxmax())
top_age      = str(df.groupby('Age Group', observed=True)['Purchase Amount (USD)'].sum().idxmax())

print("=" * 60)
print("  💡 BUSINESS INSIGHTS")
print("=" * 60)
print(f"  I01. Tổng doanh thu: ${total_rev:,.0f}")
print(f"  I02. Danh mục cao nhất: {top_category} (44.7%)")
print(f"  I03. Mùa cao điểm: {top_season} (ANOVA p=0.011 ✅)")
print(f"  I04. Sản phẩm bán chạy: {top_item}")
print(f"  I05. Địa điểm top: {top_location}")
print(f"  I06. Nhóm tuổi đóng góp nhiều nhất: {top_age}")
print(f"  I07. Subscriber KHÔNG chi tiêu cao hơn (T-test p=0.66 ❌)")
print(f"  I08. Purchase Amount phân phối đều $20-$100 → cơ hội tăng tần suất")

print()
print("=" * 60)
print("  🚀 ACTIONABLE RECOMMENDATIONS")
print("=" * 60)
recs = [
    f"R01. Tăng 30% ngân sách marketing vào mùa {top_season}",
    f"R02. Ưu tiên nhóm tuổi {top_age} trong các chiến dịch targeting",
    f"R03. Đẩy mạnh quảng bá {top_item} + {top_category}",
    f"R04. Mở rộng kênh phân phối tại {top_location}",
    "R05. Xây dựng loyalty program 4 tier để tăng tần suất mua",
    f"R06. Tăng tồn kho Clothing 40% trước mùa {top_season}",
    "R07. Cá nhân hóa email cho KH có Previous Purchases > 10",
    "R08. Phân bổ ngân sách: 44.7% Clothing, 31.8% Accessories",
]
for r in recs:
    print(f"  {r}")
